# Phase 0: Baseline Validation - Stage 1 sur Test Set Réel

## Objectif Critique

**Évaluer Stage 1 (80.5% validation) sur le vrai test set Kaggle (3,774 images)**

### Point de Décision

- ✅ **Si test accuracy ≥70%**: Procéder à Phase 1 (SMoLoRA data curation)
- ⚠️ **Si test accuracy <70%**: Domain adaptation nécessaire avant SMoLoRA

### Pourquoi C'est Critique

SMoLoRA **gèle Expert #1 (Stage 1)**. Si Stage 1 ne généralise pas au test set:
- SMoLoRA héritera de cette faiblesse
- Impossible d'atteindre ≥85% classification
- Besoin de fixer Stage 1 d'abord

### Résultats Attendus

- **Best case**: 75-80% (Stage 1 généralise bien)
- **Worst case**: <60% (distribution shift majeur)

In [1]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================
import os
import json
import re
import random
import torch
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import Counter
import numpy as np

from unsloth import FastVisionModel
from peft import PeftModel
from transformers import AutoProcessor

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch: 2.9.0+cu128
CUDA: True


In [2]:
# ============================================================
# CELL 2: CONFIGURATION
# ============================================================
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava")
NEW_IMPL_DIR = BASE_DIR / "new_implementation"

# Stage 1 checkpoint (Expert #1 pour SMoLoRA)
STAGE1_CHECKPOINT = NEW_IMPL_DIR / "checkpoints" / "stage1_cassava_qwen25vl_7b" / "final_model"

# Test set (3,774 images Kaggle)
TEST_IMAGES_DIR = BASE_DIR / "data" / "cassava" / "test" / "0"
TEST_LABELS_CSV = BASE_DIR / "data" / "sample_submission_file.csv"

MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
TARGET_IMAGE_SIZE = (512, 512)
CLASS_NAMES = ["CBB", "CBSD", "CGM", "CMD", "Healthy"]

# Label mapping CSV -> Model
LABEL_MAPPING = {
    'cbb': 'CBB',
    'cbsd': 'CBSD',
    'cgm': 'CGM',
    'cmd': 'CMD',
    'healthy': 'Healthy'
}

# Stage 1 validation results (référence)
STAGE1_VALIDATION = {
    "global": 80.52,
    "CBB": 55.13,
    "CBSD": 83.99,
    "CGM": 56.47,
    "CMD": 85.98,
    "Healthy": 75.37
}

print(f"✓ Stage 1 checkpoint: {STAGE1_CHECKPOINT}")
print(f"✓ Test images: {TEST_IMAGES_DIR}")
print(f"✓ Test labels: {TEST_LABELS_CSV}")

✓ Stage 1 checkpoint: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_qwen25vl_7b/final_model
✓ Test images: /home/hounfodjidagba/learning/cassava/data/cassava/test/0
✓ Test labels: /home/hounfodjidagba/learning/cassava/data/sample_submission_file.csv


In [3]:
# ============================================================
# CELL 3: CHARGER STAGE 1 MODEL
# ============================================================
print("Chargement du modèle de base...")
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
)

print(f"Chargement Stage 1 checkpoint: {STAGE1_CHECKPOINT}")
model = PeftModel.from_pretrained(model, str(STAGE1_CHECKPOINT))

processor = AutoProcessor.from_pretrained(str(STAGE1_CHECKPOINT))

# Mode inférence
FastVisionModel.for_inference(model)
print("✅ Stage 1 model chargé en mode inférence")

Chargement du modèle de base...
==((====))==  Unsloth 2026.1.1: Fast Qwen2_5_Vl patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Chargement Stage 1 checkpoint: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_qwen25vl_7b/final_model
✅ Stage 1 model chargé en mode inférence


In [4]:
# ============================================================
# CELL 4: CHARGER TEST SET
# ============================================================
print("Chargement du test set Kaggle...")

# Charger labels depuis CSV
test_df = pd.read_csv(TEST_LABELS_CSV)
print(f"Labels CSV: {len(test_df)} samples")

# Créer dataset de test
test_data = []
valid_samples = 0
missing_images = 0

for _, row in test_df.iterrows():
    image_path = TEST_IMAGES_DIR / row['Id']
    
    if image_path.exists():
        # Mapper label CSV vers classe modèle
        csv_label = row['Category'].lower()
        model_class = LABEL_MAPPING.get(csv_label, "Unknown")
        
        if model_class != "Unknown":
            test_data.append({
                'image_path': str(image_path),
                'image_id': row['Id'],
                'csv_label': csv_label,
                'actual': model_class
            })
            valid_samples += 1
        else:
            print(f"⚠️ Label inconnu: {csv_label} pour {row['Id']}")
    else:
        missing_images += 1

print(f"\n✅ Échantillons de test valides: {valid_samples:,}")
if missing_images > 0:
    print(f"⚠️ Images manquantes: {missing_images}")

# Distribution du test set
test_dist = Counter(sample['actual'] for sample in test_data)
print("\n📊 Distribution du test set:")
for cls in CLASS_NAMES:
    count = test_dist[cls]
    pct = count / len(test_data) * 100
    print(f"  {cls}: {count:>4} ({pct:>5.1f}%)")

Chargement du test set Kaggle...
Labels CSV: 3774 samples

✅ Échantillons de test valides: 3,774

📊 Distribution du test set:
  CBB:  753 ( 20.0%)
  CBSD:  731 ( 19.4%)
  CGM:  706 ( 18.7%)
  CMD:  800 ( 21.2%)
  Healthy:  784 ( 20.8%)


In [5]:
# ============================================================
# CELL 5: FONCTIONS D'ÉVALUATION
# ============================================================
def generate_response(image_path: str, question: str, max_new_tokens: int = 128):
    """Génère une réponse pour une image."""
    try:
        image = Image.open(image_path).convert('RGB').resize(TARGET_IMAGE_SIZE)
    except Exception as e:
        print(f"Erreur chargement image {image_path}: {e}")
        return "Erreur"

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question}
        ]
    }]

    input_text = processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = processor(
        text=[input_text], images=[[image]],
        return_tensors="pt", padding=True,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.lower().split("assistant")[-1].strip()
    return response

def extract_disease(response: str) -> str:
    """Extrait la maladie de la réponse Stage 1."""
    response_lower = response.lower()
    
    # Chercher "disease:" (format Stage 1)
    if "disease:" in response_lower:
        match = re.search(r'disease:\s*([\w\s()]+)', response_lower)
        if match:
            diag = match.group(1).strip()
            if "cmd" in diag or "mosaic" in diag:
                return "CMD"
            elif "cbsd" in diag or "brown streak" in diag:
                return "CBSD"
            elif "cbb" in diag or "bacterial" in diag:
                return "CBB"
            elif "cgm" in diag or "green" in diag:
                return "CGM"
            elif "healthy" in diag:
                return "Healthy"
    
    # Fallback - chercher mots-clés
    if "cmd" in response_lower or "mosaic" in response_lower:
        return "CMD"
    elif "cbsd" in response_lower or "brown streak" in response_lower:
        return "CBSD"
    elif "cbb" in response_lower or "bacterial" in response_lower:
        return "CBB"
    elif "cgm" in response_lower or "green mottle" in response_lower or "green mite" in response_lower:
        return "CGM"
    elif "healthy" in response_lower or "sain" in response_lower:
        return "Healthy"
    
    return "Unknown"

print("✅ Fonctions d'évaluation prêtes")

✅ Fonctions d'évaluation prêtes


In [6]:
# ============================================================
# CELL 6: ÉVALUATION SUR TEST SET COMPLET (3,774 images)
# ============================================================
print("="*70)
print("PHASE 0: ÉVALUATION STAGE 1 SUR TEST SET KAGGLE")
print("="*70)

# Question de classification
question = "What cassava disease is shown in this image?"

print(f"\n📊 Évaluation sur {len(test_data):,} images de test...")
print(f"⏱️ Temps estimé: ~2 heures (0.5s/image)")
print(f"\nQuestion: '{question}'\n")

results = []
for i, sample in enumerate(tqdm(test_data, desc="Evaluating")):
    response = generate_response(sample['image_path'], question)
    predicted = extract_disease(response)
    
    results.append({
        'image_id': sample['image_id'],
        'actual': sample['actual'],
        'predicted': predicted,
        'correct': predicted == sample['actual'],
        'response': response[:200]
    })
    
    # Affichage du progrès tous les 200 échantillons
    if (i + 1) % 200 == 0:
        current_acc = sum(1 for r in results if r['correct']) / len(results) * 100
        print(f"\n📊 Progrès {i+1:,}/{len(test_data):,}: Accuracy actuelle = {current_acc:.1f}%")

print("\n✅ Évaluation terminée!")

PHASE 0: ÉVALUATION STAGE 1 SUR TEST SET KAGGLE

📊 Évaluation sur 3,774 images de test...
⏱️ Temps estimé: ~2 heures (0.5s/image)

Question: 'What cassava disease is shown in this image?'



Evaluating:   5%|▌         | 200/3774 [10:24<3:06:27,  3.13s/it]


📊 Progrès 200/3,774: Accuracy actuelle = 15.5%


Evaluating:   7%|▋         | 252/3774 [13:03<3:02:36,  3.11s/it]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 7: RÉSULTATS & POINT DE DÉCISION
# ============================================================
correct = sum(1 for r in results if r['correct'])
test_accuracy = correct / len(results) * 100

print("\n" + "="*70)
print("RÉSULTATS STAGE 1 - TEST SET KAGGLE (3,774 images)")
print("="*70)

print(f"\n🎯 Accuracy globale:")
print(f"  - Test set: {test_accuracy:.2f}%")
print(f"  - Validation (référence): {STAGE1_VALIDATION['global']:.2f}%")
val_diff = test_accuracy - STAGE1_VALIDATION['global']
print(f"  - Différence: {val_diff:+.2f}%")

# Par classe
print(f"\n📈 Métriques par classe:")
print(f"{'Classe':<10} | {'Test':<8} | {'Val (réf)':<12} | {'Diff':<8} | {'Samples':<8} | Status")
print("-" * 75)

class_metrics = {}
for cls in CLASS_NAMES:
    cls_samples = [r for r in results if r['actual'] == cls]
    if cls_samples:
        cls_correct = sum(1 for r in cls_samples if r['correct'])
        cls_acc = cls_correct / len(cls_samples) * 100
        class_metrics[cls] = {
            'test_acc': cls_acc,
            'val_acc': STAGE1_VALIDATION.get(cls, 0),
            'correct': cls_correct,
            'total': len(cls_samples)
        }
        
        val_acc = STAGE1_VALIDATION.get(cls, 0)
        cls_diff = cls_acc - val_acc
        
        # Status
        if cls_acc >= 70:
            status = "✅"
        elif cls_acc >= 60:
            status = "⚠️"
        else:
            status = "❌"
        
        print(f"{cls:<10} | {cls_acc:>6.1f}% | {val_acc:>10.1f}% | {cls_diff:>+6.1f}% | {len(cls_samples):>6} | {status}")
    else:
        print(f"{cls:<10} | {'N/A':<8} | {'N/A':<12} | {'N/A':<8} | {'0':<8} | ⚠️")

In [ ]:
# ============================================================
# CELL 8: POINT DE DÉCISION - PROCÉDER OU NON
# ============================================================
print("\n" + "="*70)
print("🎯 POINT DE DÉCISION - PHASE 0")
print("="*70)

# Décision basée sur accuracy test
if test_accuracy >= 75:
    decision = "EXCELLENT"
    proceed = True
    message = f"""
🎉 EXCELLENT! Stage 1 généralise bien au test set ({test_accuracy:.1f}%)

✅ **DÉCISION: PROCÉDER À PHASE 1 (SMoLoRA Data Curation)**

Raisons:
- Test accuracy {test_accuracy:.1f}% ≥ 75% (seuil excellent)
- Bonne généralisation validation → test
- Expert #1 (Stage 1 gelé) fournira base solide pour SMoLoRA

Prochaines étapes:
1. Phase 1: Curation du dataset VQA (15k samples)
2. Phase 2: Implémentation SMoLoRA
3. Phase 3: Training Expert #2 + Router
"""
    
elif test_accuracy >= 70:
    decision = "ACCEPTABLE"
    proceed = True
    message = f"""
✅ ACCEPTABLE! Stage 1 généralise au test set ({test_accuracy:.1f}%)

✅ **DÉCISION: PROCÉDER À PHASE 1 (SMoLoRA Data Curation)**

Raisons:
- Test accuracy {test_accuracy:.1f}% ≥ 70% (seuil minimum)
- Généralisation acceptable
- SMoLoRA peut améliorer avec Expert #2 (VQA)

Notes:
- Attention aux classes faibles identifiées ci-dessus
- SMoLoRA training pourrait nécessiter plus d'itérations

Prochaines étapes:
1. Phase 1: Curation du dataset VQA (15k samples)
2. Phase 2: Implémentation SMoLoRA
"""
    
elif test_accuracy >= 60:
    decision = "LIMITE"
    proceed = False
    message = f"""
⚠️ LIMITE! Distribution shift détecté ({test_accuracy:.1f}% vs {STAGE1_VALIDATION['global']:.1f}% validation)

❌ **DÉCISION: NE PAS PROCÉDER - DOMAIN ADAPTATION NÉCESSAIRE**

Problème:
- Test accuracy {test_accuracy:.1f}% < 70% (seuil minimum)
- Écart {val_diff:.1f}% validation → test indique overfitting
- SMoLoRA gelère Expert #1 → héritera de cette faiblesse

Actions requises AVANT SMoLoRA:
1. Analyser distribution shift (voir cellule suivante)
2. Appliquer domain adaptation:
   - Data augmentation ciblée sur classes faibles
   - Retrain Stage 1 avec test-like validation split
   - Test-Time Augmentation (TTA)
3. Re-évaluer sur test set
4. Si test accuracy ≥ 70%, revenir à Phase 1
"""
    
else:
    decision = "CRITIQUE"
    proceed = False
    message = f"""
❌ CRITIQUE! Distribution shift majeur ({test_accuracy:.1f}% vs {STAGE1_VALIDATION['global']:.1f}% validation)

❌ **DÉCISION: NE PAS PROCÉDER - RETRAIN STAGE 1 NÉCESSAIRE**

Problème critique:
- Test accuracy {test_accuracy:.1f}% << 60% (inacceptable)
- Écart {val_diff:.1f}% indique overfitting sévère
- Validation set ne représente pas test set

Actions requises:
1. **RETRAIN Stage 1 complètement:**
   - Créer Kaggle-only validation split (20% du test set)
   - Data augmentation aggressive
   - Curriculum learning
   - Focal loss pour classes déséquilibrées
2. Objectif: Test accuracy ≥ 75%
3. Après retraining, revenir à Phase 0

Alternative:
- Si retraining impossible, considérer classification-only (pas VQA)
"""

print(message)

# Sauvegarder décision
decision_result = {
    'test_accuracy': test_accuracy,
    'validation_accuracy': STAGE1_VALIDATION['global'],
    'difference': val_diff,
    'decision': decision,
    'proceed_to_phase1': proceed,
    'class_metrics': class_metrics
}

# Sauvegarder résultats
results_dir = NEW_IMPL_DIR / "results" / "phase0_baseline"
results_dir.mkdir(parents=True, exist_ok=True)

with open(results_dir / "stage1_test_results.json", 'w') as f:
    json.dump(decision_result, f, indent=2)

print(f"\n💾 Résultats sauvegardés: {results_dir / 'stage1_test_results.json'}")

In [ ]:
# ============================================================
# CELL 9: ANALYSE DISTRIBUTION SHIFT (si test < validation)
# ============================================================
if test_accuracy < STAGE1_VALIDATION['global']:
    print("\n" + "="*70)
    print("🔍 ANALYSE DISTRIBUTION SHIFT")
    print("="*70)
    
    print("\n📊 Comparaison Validation vs Test par classe:")
    print(f"{'Classe':<10} | {'Val Acc':<10} | {'Test Acc':<10} | {'Shift':<10} | Analyse")
    print("-" * 80)
    
    for cls in CLASS_NAMES:
        if cls in class_metrics:
            val_acc = class_metrics[cls]['val_acc']
            test_acc = class_metrics[cls]['test_acc']
            shift = test_acc - val_acc
            
            if shift < -20:
                analysis = "CRITIQUE - Shift majeur"
            elif shift < -10:
                analysis = "ATTENTION - Distribution différente"
            elif shift < -5:
                analysis = "Léger shift"
            elif shift > 10:
                analysis = "Amélioration (suspect)"
            else:
                analysis = "Normal"
            
            print(f"{cls:<10} | {val_acc:>8.1f}% | {test_acc:>9.1f}% | {shift:>+8.1f}% | {analysis}")
    
    # Matrice de confusion
    print("\n📊 Matrice de confusion:")
    confusion = {}
    for cls in CLASS_NAMES:
        confusion[cls] = Counter(r['predicted'] for r in results if r['actual'] == cls)
    
    print(f"\n{'Actual':<10} | " + " | ".join(f"{c:<8}" for c in CLASS_NAMES + ["Unknown"]))
    print("-" * 85)
    for actual in CLASS_NAMES:
        row_counts = []
        for pred in CLASS_NAMES + ["Unknown"]:
            count = confusion[actual].get(pred, 0)
            row_counts.append(f"{count:<8}")
        print(f"{actual:<10} | " + " | ".join(row_counts))
    
    # Classes les plus confondues
    print("\n🔀 Top 5 confusions:")
    confusions_list = []
    for actual in CLASS_NAMES:
        for pred, count in confusion[actual].most_common():
            if pred != actual and pred != "Unknown":
                confusions_list.append((actual, pred, count))
    
    confusions_list.sort(key=lambda x: x[2], reverse=True)
    for i, (actual, pred, count) in enumerate(confusions_list[:5]):
        print(f"  {i+1}. {actual} → {pred}: {count} erreurs")
else:
    print("\n✅ Pas de distribution shift détecté (test ≥ validation)")

In [ ]:
# ============================================================
# CELL 10: EXEMPLES DE PRÉDICTIONS
# ============================================================
print("\n" + "="*70)
print("📝 EXEMPLES DE PRÉDICTIONS")
print("="*70)

# 5 exemples corrects
correct_samples = [r for r in results if r['correct']]
if correct_samples:
    print("\n✅ 5 exemples CORRECTS:")
    for i, sample in enumerate(random.sample(correct_samples, min(5, len(correct_samples)))):
        print(f"\n[{i+1}] {sample['actual']} → {sample['predicted']} ✅")
        print(f"    Réponse: {sample['response'][:150]}...")

# 5 exemples incorrects
incorrect_samples = [r for r in results if not r['correct']]
if incorrect_samples:
    print("\n\n❌ 5 exemples INCORRECTS:")
    for i, sample in enumerate(random.sample(incorrect_samples, min(5, len(incorrect_samples)))):
        print(f"\n[{i+1}] {sample['actual']} → {sample['predicted']} ❌")
        print(f"    Réponse: {sample['response'][:150]}...")

## Conclusion Phase 0

### Résumé

Cette évaluation a testé Stage 1 (Expert #1 pour SMoLoRA) sur le vrai test set Kaggle.

### Décision

Voir **CELL 8** pour la décision finale:
- ✅ **Si test accuracy ≥70%**: Procéder à Phase 1 (SMoLoRA)
- ❌ **Si test accuracy <70%**: Domain adaptation nécessaire d'abord

### Prochaines Étapes

**Si PROCÉDER:**
1. Phase 1: Créer `12b_curate_vqa_data.ipynb`
2. Nettoyer Stage 2 dataset (29k → 15k curated)
3. Phase 2: Implémenter SMoLoRA architecture

**Si NE PAS PROCÉDER:**
1. Analyser distribution shift (CELL 9)
2. Appliquer domain adaptation
3. Retrain Stage 1 si nécessaire
4. Revenir à Phase 0